In [ ]:


## Juego deportivo, "Fútbol de Duelo Verbal".

Aquí, la habilidad no estará en tus reflejos con el joystick, sino en tu ingenio, tu memoria y tu capacidad para elegir la "jugada" verbal correcta en el momento justo.



1. Discusión: De Duelos de Espadas a Duelos en el Campo
¿Cómo traducimos la mecánica?

La Espada es el Balón: El jugador que tiene el balón es el atacante, el que "lanza el insulto". El defensor cercano es quien debe "responder a la ofensa".

"Insulto" = Jugada de Ataque: La frase del atacante no es un insulto literal, sino una descripción florida y jactanciosa de la jugada que va a intentar.

Ejemplo: "¡Voy a hacer un túnel tan grande que podrás aparcar un autobús dentro!"

"Réplica" = Maniobra Defensiva: La respuesta del defensor es la acción táctica para contrarrestar la jugada. De varias opciones, solo una es la correcta.

Ejemplo de réplica correcta: "Inténtalo, pero mis piernas están más cerradas que la caja fuerte de un banco."

Ejemplo de réplica incorrecta: "¡Pues la matrícula de mi autobús es 'G-O-L'!" (El atacante te supera).

El Flujo del Partido: El juego es por turnos. Un equipo ataca. El jugador con el balón se enfrenta a un defensor, iniciando un duelo. Si el atacante gana el duelo, avanza en el campo. Si el defensor gana, roba el balón. Después de varios duelos ganados, el atacante llega al área y se enfrenta al portero en un duelo final y más difícil para marcar el gol.

El Rol de los Jugadores y sus Estadísticas:

Los jugadores reales que mencionas no tendrán velocidad o fuerza, sino atributos mentales:

Creatividad (Ataque): Determina la calidad y variedad de las "frases de ataque" que un jugador puede usar. Un Messi o un Ronaldinho tendrían un repertorio de frases casi imbatible.

Anticipación (Defensa): Afecta la cantidad de "opciones de réplica" que se le presentan al defensor. Un buen defensor como Maldini podría tener solo 2 opciones (50% de acierto), mientras que un delantero defendiendo podría tener 4 o 5, haciéndolo más difícil.

Temple (Finalización): Afecta la dificultad del duelo final contra el portero.

2. Estructura de Archivos y Datos
Este nuevo proyecto tendrá su propia carpeta y, de forma crucial, archivos de datos para los equipos y los diálogos.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- challenge.py         # ¡NUEVO! Nuestro motor de duelos verbales.
|
|-- insult_soccer/           # ¡NUEVO PROYECTO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- player_teamA.png
|   |   |-- player_teamB.png
|   |   |-- field.png
|   |
|   |-- data/                # ¡NUEVO!
|   |   |-- teams.json
|   |   |-- challenges.json
|   |
|   |-- scenes/
|   |   |-- __init__.py
|   |   |-- match_scene.py
|   |
|   |-- main.py
|
|-- requirements.txt
insult_soccer/data/teams.json (Ejemplo)
JSON

{
  "equipos": [
    {
      "nombre": "Titanes de la Capital",
      "jugadores": [
        {"nombre": "Leo 'El Mago'", "creatividad": 95, "anticipacion": 40, "temple": 92},
        {"nombre": "Sergio 'El Muro'", "creatividad": 30, "anticipacion": 98, "temple": 60}
      ]
    },
    {
      "nombre": "Furia del Interior",
      "jugadores": [
        {"nombre": "Carlos 'El Cohete'", "creatividad": 91, "anticipacion": 50, "temple": 85},
        {"nombre": "Roberto 'El Káiser'", "creatividad": 40, "anticipacion": 95, "temple": 70}
      ]
    }
  ]
}
insult_soccer/data/challenges.json (Ejemplo)
JSON

{
  "duelos": [
    {
      "id": 1,
      "ataque": "Voy a driblarte con la elegancia de un bailarín de tango.",
      "replica": "Hasta el mejor bailarín tropieza si le cambian el suelo.",
      "fallos": ["¡Pero esto es fútbol, no un salón de baile!", "Mi especialidad es bailar sobre tus tobillos."]
    },
    {
      "id": 2,
      "ataque": "Mi disparo tiene más efecto que tu café de la mañana.",
      "replica": "No importa el efecto si el portero es un muro de hormigón.",
      "fallos": ["¡Yo tomo té, no café!", "Pues a mí me gusta el café amargo."]
    }
  ]
}
3. El Código del Motor Actualizado
Añadimos un nuevo módulo para manejar estos duelos.

engine/challenge.py (¡NUEVO!)
Python

# engine/challenge.py
import random

class ChallengeManager:
    def __init__(self, challenges_data):
        self.challenges = challenges_data['duelos']
        self.current_challenge = None
        self.options = []

    def start_random_challenge(self):
        """Inicia un nuevo duelo aleatorio."""
        self.current_challenge = random.choice(self.challenges)

        # Preparamos las opciones para el defensor
        correct_answer = self.current_challenge['replica']
        wrong_answers = self.current_challenge['fallos']

        self.options = [correct_answer] + wrong_answers
        random.shuffle(self.options)

        return self.current_challenge['ataque'], self.options

    def resolve_choice(self, chosen_option):
        """Comprueba si la opción elegida es la réplica correcta."""
        if not self.current_challenge:
            return False
        return chosen_option == self.current_challenge['replica']

4. El Código del Juego de Duelo Verbal
insult_soccer/scenes/match_scene.py (¡NUEVO!)
Esta es la escena principal donde ocurre la magia.

Python

# insult_soccer/scenes/match_scene.py
import pygame
import json
from engine.scene import Scene
from engine.challenge import ChallengeManager

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)

        # Carga de datos
        with open('insult_soccer/data/challenges.json', 'r', encoding='utf-8') as f:
            challenges_data = json.load(f)
        self.challenge_manager = ChallengeManager(challenges_data)

        # Estado del juego
        self.game_state = "new_challenge" # 'new_challenge', 'player_defending', 'result'
        self.attacker_phrase = ""
        self.defender_options = []
        self.result_text = ""

        self.font = pygame.font.Font(None, 32)
        self.option_rects = []

    def update(self):
        if self.game_state == "new_challenge":
            self.attacker_phrase, self.defender_options = self.challenge_manager.start_random_challenge()
            self.game_state = "player_defending"

    def handle_events(self, events):
        if self.game_state == "player_defending":
            for event in events:
                if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    for i, rect in enumerate(self.option_rects):
                        if rect.collidepoint(event.pos):
                            chosen_option = self.defender_options[i]
                            is_correct = self.challenge_manager.resolve_choice(chosen_option)
                            if is_correct:
                                self.result_text = "¡Defensa exitosa! Recuperas el balón."
                            else:
                                self.result_text = "¡Te han superado! El rival avanza."
                            self.game_state = "result"
                            break

        elif self.game_state == "result":
            for event in events:
                if event.type == pygame.KEYDOWN or event.type == pygame.MOUSEBUTTONDOWN:
                    self.game_state = "new_challenge"


    def draw(self, surface):
        surface.fill((50, 150, 50)) # Verde campo de fútbol

        # Dibujar la frase del atacante
        attacker_surf = self.font.render(f"RIVAL: '{self.attacker_phrase}'", True, (255, 255, 255))
        surface.blit(attacker_surf, (50, 100))

        if self.game_state == "player_defending":
            # Dibujar las opciones del defensor
            self.option_rects = []
            for i, option in enumerate(self.defender_options):
                option_surf = self.font.render(f"{i+1}. {option}", True, (220, 220, 0))
                option_rect = option_surf.get_rect(topleft=(70, 200 + i * 50))
                self.option_rects.append(option_rect)
                surface.blit(option_surf, option_rect)

        elif self.game_state == "result":
            result_surf = self.font.render(self.result_text, True, (255, 255, 255))
            result_rect = result_surf.get_rect(center=(surface.get_width()/2, surface.get_height()/2))
            surface.blit(result_surf, result_rect)
Clase 24: Fusión de Géneros - El Duelo Verbal Deportivo
Contenidos de la Clase
Discusión sobre Fusión de Géneros (20 min): Iniciar la clase discutiendo cómo la combinación de mecánicas de géneros aparentemente opuestos puede dar lugar a juegos innovadores y memorables. Usar el concepto de "Fútbol de Duelo Verbal" como caso de estudio.

Diseñando la Base de Datos (30 min):

Explicar la importancia de externalizar los datos.

Diseñar y crear los archivos teams.json y challenges.json. Discutir cómo la estructura de estos archivos define las posibilidades del juego. Enseñar a los alumnos a añadir sus propios equipos y duelos.

El Motor de Desafíos (35 min):

Implementar la clase ChallengeManager en el motor.

Explicar su función: es un sistema agnóstico al juego que puede gestionar cualquier tipo de duelo "pregunta-respuesta". Podría reutilizarse para un concurso de Trivial, por ejemplo.

Detallar la lógica de start_random_challenge y cómo baraja las opciones para que la respuesta correcta no esté siempre en el mismo sitio.

Implementando la Escena del Partido (35 min):

Crear la MatchScene, que será el núcleo del juego.

Implementar la máquina de estados simple (new_challenge, player_defending, result).

Mostrar cómo se usa el ChallengeManager para obtener los datos del duelo y cómo se gestiona la interacción del jugador (clic en las opciones) para resolver el desafío.

Cierre y Próximos Pasos
Resumen: ¡Hemos dado un paso increíblemente creativo! Hemos diseñado y comenzado a implementar un juego que fusiona la estrategia de una aventura gráfica con la temática del deporte. Hemos creado un motor de desafíos reutilizable y una estructura de datos que nos permite expandir el juego fácilmente con nuevos equipos y diálogos.

Adelanto de la Clase 25: "Tenemos el duelo individual. Ahora, tenemos que crear el 'partido'. En la próxima clase, daremos vida al campo de juego. Posicionaremos los sprites de los jugadores de ambos equipos en el campo. Implementaremos un sistema de turnos y posesión. El resultado de un duelo ya no será un simple texto, sino que provocará que el 'balón' (la posesión) se mueva de un jugador a otro en el campo, conectando los duelos para crear una jugada completa hasta llegar al área rival."






vamos por la siguiente clase



Ver razonamiento
¡Perfecto! Avancemos. Es hora de sacar nuestro juego del abstracto menú de duelo y ponerlo sobre el césped.

En esta clase, daremos vida al partido. Conectaremos nuestra mecánica de duelo verbal con una representación visual en el campo de juego. Los jugadores ya no serán solo nombres, sino sprites que se mueven, se enfrentan y luchan por la posesión del balón, creando una verdadera progresión táctica hacia la portería.

1. Discusión: El Tablero de Juego Deportivo
Muchos de los mejores juegos de estrategia deportiva no se basan en la acción en tiempo real. En su lugar, tratan el campo de juego como un tablero, similar al ajedrez o a un juego de rol táctico.

El Espacio y las Zonas: El campo se divide en zonas (defensa, mediocampo, ataque). El objetivo es mover la "pieza" que tiene el balón de una zona a la siguiente, superando los desafíos que se presentan.

Encuentros por Turnos: Cuando una pieza atacante (con el balón) entra en el radio de acción de una pieza defensora, se desencadena un "encuentro". En nuestro caso, este encuentro es un duelo verbal. El resto del tablero "espera" el resultado de este encuentro individual.

Resolución y Consecuencia: El resultado del encuentro tiene una consecuencia directa y espacial. Ganar significa avanzar a la siguiente zona. Perder significa que la posesión (el balón) pasa a la pieza defensora, y los roles se invierten.

Ejemplos en juegos conocidos:

Football Manager ⚽: El ejemplo definitivo. El partido se desarrolla a través de estadísticas, posiciones y eventos clave. No controlas directamente el disparo, sino que pones a tus mejores piezas en las mejores posiciones para que la simulación resuelva el encuentro a tu favor.

Blood Bowl 🏈: Una versión de fantasía del fútbol americano que es, en esencia, un juego de mesa por turnos sobre una rejilla. Cada acción (mover, placar, pasar) es un "encuentro" que se resuelve con una tirada de dados, y el resultado determina el nuevo estado del tablero.

Final Fantasy Tactics / XCOM ♟️: Aunque no son juegos deportivos, su mecánica es la misma que buscamos. Mueves tus unidades en un tablero por turnos. Cuando un enemigo está al alcance, inicias un "encuentro" (un ataque). El resultado (acierto o fallo) determina el siguiente movimiento.

2. Estructura de Archivos Actualizada
No necesitamos nuevos módulos en el motor, pero sí un nuevo archivo para nuestras entidades de juego y un asset para el campo.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- insult_soccer/
|   |-- assets/
|   |   |-- field.png        # ¡NUEVO! Imagen del campo de fútbol 5.
|   |
|   |-- data/
|   |   |-- ... (sin cambios)
|   |
|   |-- entities/            # ¡NUEVO!
|   |   |-- __init__.py
|   |   |-- player.py
|   |
|   |-- scenes/
|   |   |-- ... (sin cambios)
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Juego
insult_soccer/entities/player.py (¡NUEVO!)
Creamos una clase para nuestro jugador, que contendrá tanto sus datos (nombre, stats) como su representación visual (sprite).

Python

# insult_soccer/entities/player.py
import pygame
from engine.game_object import GameObject

class SoccerPlayer(GameObject):
    def __init__(self, x, y, image_path, player_data):
        super().__init__(x, y, image_path)

        self.player_data = player_data
        self.nombre = player_data['nombre']
        self.creatividad = player_data['creatividad']
        self.anticipacion = player_data['anticipacion']
        self.temple = player_data['temple']

        self.is_selected = False # Para resaltar al jugador activo

    def draw(self, surface):
        # Sobreescribimos draw para añadir un resaltado si está seleccionado
        super().draw(surface)
        if self.is_selected:
            pygame.draw.rect(surface, (255, 255, 0), self.rect, 2) # Borde amarillo
insult_soccer/scenes/match_scene.py (Grandes Cambios)
Esta escena se vuelve mucho más compleja. Ahora gestiona la posición de los jugadores, la posesión del balón y el flujo del partido.

Python

# insult_soccer/scenes/match_scene.py
import pygame
import json
from engine.scene import Scene
from engine.challenge import ChallengeManager
from ..entities.player import SoccerPlayer

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        self.load_data()
        self.setup_field()
        self.start_match()

    def load_data(self):
        with open('insult_soccer/data/teams.json', 'r', encoding='utf-8') as f:
            self.teams_data = json.load(f)['equipos']
        with open('insult_soccer/data/challenges.json', 'r', encoding='utf-8') as f:
            challenges_data = json.load(f)
        self.challenge_manager = ChallengeManager(challenges_data)
        self.background = pygame.image.load('insult_soccer/assets/field.png').convert()

    def setup_field(self):
        """Crea y posiciona a los jugadores en el campo."""
        self.todos_los_sprites = pygame.sprite.Group()
        self.team_a_players = pygame.sprite.Group()
        self.team_b_players = pygame.sprite.Group()

        # Posiciones fijas para un 2-1-1 (Portero, 2 Def, 1 Med, 1 Del)
        # Equipo A (izquierda), Equipo B (derecha)
        positions_a = [(50, 300), (200, 150), (200, 450), (350, 300), (450, 300)]
        positions_b = [(750, 300), (600, 150), (600, 450), (450, 300), (350, 300)]

        team_a_data = self.teams_data[0]['jugadores']
        for i, pos in enumerate(positions_a):
            player = SoccerPlayer(pos[0], pos[1], 'insult_soccer/assets/player_teamA.png', team_a_data[i])
            self.team_a_players.add(player)

        # ... (hacer lo mismo para el equipo B) ...
        self.todos_los_sprites.add(self.team_a_players, self.team_b_players)

    def start_match(self):
        """Inicia el estado del partido."""
        self.possession_team = self.team_a_players # Empieza el equipo A
        self.player_with_ball = self.team_a_players.sprites()[0] # El portero
        self.player_with_ball.is_selected = True
        self.game_state = "player_move" # 'player_move', 'challenge', 'result'
        # ... (resto de la lógica de estado) ...

    def start_new_challenge(self):
        """Inicia un duelo entre el atacante y un defensor cercano."""
        self.player_with_ball.is_selected = True
        # Lógica para encontrar al defensor más cercano
        self.current_defender = self.find_nearest_defender()
        self.current_defender.is_selected = True

        self.attacker_phrase, self.defender_options = self.challenge_manager.start_random_challenge()
        self.game_state = "challenge"

    def resolve_challenge(self, success):
        self.player_with_ball.is_selected = False
        self.current_defender.is_selected = False

        if success:
            self.result_text = "¡Avance exitoso!"
            # Lógica para mover el balón al siguiente jugador del equipo atacante
            # self.player_with_ball = find_next_attacker(self.player_with_ball)
        else:
            self.result_text = "¡Balón robado!"
            # El defensor ahora tiene el balón, se invierte la posesión
            self.possession_team = self.team_b_players # o el equipo que corresponda
            self.player_with_ball = self.current_defender

        self.game_state = "result"

    def update(self):
        if self.game_state == "player_move":
            # En un juego más complejo, aquí el jugador elegiría a quién pasar
            # Por ahora, iniciamos un desafío automáticamente
            self.start_new_challenge()

    def handle_events(self, events):
        if self.game_state == "challenge":
            # ... (lógica de clic en las opciones, como en la clase anterior) ...
            # Al resolver, llamar a self.resolve_challenge(is_correct)
            pass
        elif self.game_state == "result":
            # ... (esperar clic para continuar) ...
            # self.game_state = "player_move"
            pass

    def draw(self, surface):
        surface.blit(self.background, (0, 0))
        self.todos_los_sprites.draw(surface)

        if self.game_state == "challenge":
            # ... (dibujar la UI del duelo) ...
            pass
        elif self.game_state == "result":
            # ... (dibujar el texto del resultado) ...
            pass
Clase 25: El Partido - Posesión, Turnos y Movimiento en el Campo
Contenidos de la Clase
Discusión sobre el "Tablero Deportivo" (20 min): Iniciar con la discusión sobre cómo los juegos de estrategia representan la acción deportiva, usando los ejemplos de Football Manager y Blood Bowl.

La Entidad Jugador (30 min):

Crear la clase SoccerPlayer, que combina un GameObject con los datos cargados del JSON (nombre, stats).

Implementar una forma simple de feedback visual, como un borde de resaltado, para indicar qué jugadores están activos en el duelo actual.

Montando el Escenario (35 min):

En MatchScene, implementar el método setup_field. Cargar los datos de los equipos desde el JSON y crear una instancia de SoccerPlayer para cada uno, posicionándolos en una formación táctica fija en el campo.

Introducir el concepto de posesión. Crear variables como self.possession_team y self.player_with_ball para rastrear quién tiene el control.

Conectando Duelos y Jugadas (35 min):

Refinar la máquina de estados para reflejar el flujo de un partido: player_move (decidir la siguiente acción), challenge (resolver el duelo), result (ver la consecuencia).

Implementar la lógica clave en resolve_challenge:

Si el atacante gana, la posesión se mueve a un compañero de equipo más adelantado en el campo.

Si el defensor gana, la posesión cambia de equipo, y el defensor se convierte en el nuevo atacante.

Mostrar cómo el resultado del duelo ahora tiene una consecuencia espacial y táctica en el tablero de juego.

Cierre y Próximos Pasos
Resumen: ¡Nuestro juego ha cobrado vida! Hemos pasado de un simple duelo de frases a un partido táctico por turnos. Hemos poblado el campo con jugadores, implementado un sistema de posesión y conectado los resultados de los duelos verbales a un movimiento estratégico en el campo. El juego ahora tiene un flujo y un objetivo claros.

Adelanto de la Clase 26: "Hemos llegado a la zona de ataque. ¿Y ahora qué? En la próxima clase, implementaremos la jugada final: el tiro a puerta. Este será un duelo especial y más difícil contra el portero. También añadiremos un marcador, un sistema de goles y la lógica para reiniciar el partido después de un gol, completando así el bucle de juego y convirtiendo nuestro prototipo en un juego completo de principio a fin."






perfecto! avanza, e introduce estas variantes si te parece, cada equipo tiene 5 jugadores, que los jugaores se puedan mover con las flechas, y haya un botón para cambiar de jugador seleccionado por quipo, otro para pasar la pelota, otro para marcar y otro para tirar, el de marcar sirve para atajar también, agrega una zona, la portería. y que además de los atributos que definiste por jugador, haya un atributo de área circular, cuanto mayor sea el área de un marcador, antes podra intentar bloquear al rival.




Ver razonamiento
¡Entendido! Esto es un giro fascinante y ambicioso. Abandonamos el sistema por turnos para sumergirnos en la acción en tiempo real, pero manteniendo la esencia de "rol" donde las estadísticas de los jugadores deciden el resultado de las acciones.

Me parece una evolución excelente. Estamos creando un género híbrido que podríamos llamar "Fútbol de Rol Táctico en Tiempo Real". El jugador tiene el control directo del movimiento, pero el éxito de un pase, un tiro o una entrada no depende de su habilidad para pulsar un botón en el milisegundo exacto, sino de las estadísticas del jugador que controla.

Manos a la obra. Adaptaremos nuestro motor y nuestro juego a esta nueva y emocionante visión.

1. Discusión: El Híbrido de Acción y Rol
Este nuevo modelo combina lo mejor de dos mundos:

Control Directo (Acción): El jugador se siente en control. Mover a su jugador favorito con las flechas, posicionarse para un pase o una entrada, crea una conexión inmediata y visceral con el juego.

Resolución por Estadísticas (Rol): Aquí es donde nuestro diseño original sigue vivo. Cuando pulsas el botón de "pasar", no eres tú quien define la precisión, es la estadística de Creatividad de tu jugador. Cuando intentas una entrada, tu éxito depende de tu Anticipación contra la Creatividad del rival. Esto hace que la elección del jugador y la táctica del equipo sigan siendo fundamentales.

¿Cómo se readaptan nuestras estadísticas?

Creatividad (Ataque): Ahora determina la calidad de las acciones ofensivas.

Pase: Un jugador con alta Creatividad dará pases más precisos y rápidos, más difíciles de interceptar.

Tiro: Afecta la precisión del disparo a puerta.

Anticipación (Defensa): Se convierte en el área de influencia que propusiste. Un jugador con alta Anticipación tiene un círculo de marcaje más grande. Si el rival con el balón entra en ese círculo, el defensor puede intentar una entrada.

Temple (Duelos Clave): Se usa en momentos de máxima presión.

Tiro a Puerta: El Temple del delantero se enfrenta al Temple del portero para determinar la probabilidad de gol.

Entradas Críticas: Podría usarse en un duelo 50/50 por un balón suelto.

2. Estructura de Archivos Actualizada
Reflejamos el nuevo género en el nombre del proyecto y añadimos una entidad para el balón.

/curso_pygame/
|
|-- engine/
|   |-- ... (sin cambios)
|
|-- soccer_rpg/              # ¡PROYECTO RENOMBRADO!
|   |-- __init__.py
|   |-- assets/
|   |   |-- ball.png         # ¡NUEVO!
|   |
|   |-- entities/
|   |   |-- __init__.py
|   |   |-- player.py
|   |   |-- ball.py          # ¡NUEVO!
|   |
|   |-- scenes/
|   |   |-- ... (sin cambios)
|   |
|   |-- main.py
|
|-- requirements.txt
3. El Código del Juego Actualizado
soccer_rpg/entities/ball.py (¡NUEVO!)
El balón ahora es su propia entidad, controlada por físicas para un movimiento suave.

Python

# soccer_rpg/entities/ball.py
import pygame
import pymunk
from engine.physics_object import PhysicsObject

class Ball(PhysicsObject):
    def __init__(self, x, y, space):
        super().__init__(x, y, 'soccer_rpg/assets/ball.png', space, mass=0.5, radius=8, elasticity=0.7)
        self.body.damping = 0.97 # Fricción con el césped
        self.carrier = None # Jugador que lleva el balón

    def update(self):
        super().update()
        # Si un jugador lleva el balón, la posición del balón es la del jugador
        if self.carrier:
            self.body.position = self.carrier.rect.center
            self.body.velocity = 0, 0 # El balón no se mueve por sí mismo
soccer_rpg/entities/player.py (Grandes Cambios)
El jugador ahora es mucho más complejo, con lógica de movimiento y acciones.

Python

# soccer_rpg/entities/player.py
import pygame
from engine.game_object import GameObject

class SoccerPlayer(GameObject):
    def __init__(self, x, y, image_path, player_data, team):
        super().__init__(x, y, image_path)

        self.player_data = player_data
        self.nombre = player_data['nombre']
        self.creatividad = player_data['creatividad']
        self.anticipacion = player_data['anticipacion']
        self.temple = player_data['temple']
        self.team = team

        self.velocidad = 4
        # El área de marcaje depende de la Anticipación
        self.radio_marcaje = 20 + (self.anticipacion / 10)

    def move(self, dx, dy):
        self.rect.x += dx * self.velocidad
        self.rect.y += dy * self.velocidad

    def shoot(self, ball, goal_pos):
        # La probabilidad de éxito depende del Temple vs. el del portero (lógica futura)
        print(f"{self.nombre} chuta a puerta!")
        ball.carrier = None
        fuerza = 20 + (self.temple / 10)
        # ... (código para aplicar impulso al cuerpo físico del balón) ...

    def pass_ball(self, ball, target_player):
        # La precisión depende de la Creatividad
        print(f"{self.nombre} pasa el balón a {target_player.nombre}")
        ball.carrier = None
        # ... (código para aplicar impulso al balón hacia el compañero) ...

    def tackle(self, target_player):
        # La probabilidad de éxito depende de la Anticipación vs. Creatividad del rival
        print(f"{self.nombre} intenta robar el balón a {target_player.nombre}")
        # ... (lógica de resolución del robo) ...

    def draw(self, surface, is_active):
        super().draw(surface)
        if is_active:
            pygame.draw.rect(surface, (255, 255, 0), self.rect, 2)

        # Dibujar el radio de marcaje si no tiene el balón
        if not is_active:
            pygame.draw.circle(surface, self.team, self.rect.center, self.radio_marcaje, 1)

soccer_rpg/scenes/match_scene.py (Grandes Cambios)
La escena ahora es un bucle de juego en tiempo real que gestiona la selección de jugador y las acciones.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ... (cargar datos, crear espacio de físicas) ...

        self.active_player = self.team_a_players.sprites()[0]
        self.ball = Ball(self.active_player.rect.centerx, self.active_player.rect.centery, self.space)
        self.ball.carrier = self.active_player
        self.todos_los_sprites.add(self.ball)

    def handle_events(self, events):
        # Este método ahora solo gestiona los inputs del teclado
        keys = pygame.key.get_pressed()

        # Movimiento del jugador activo
        dx, dy = 0, 0
        if keys[pygame.K_LEFT]: dx = -1
        if keys[pygame.K_RIGHT]: dx = 1
        if keys[pygame.K_UP]: dy = -1
        if keys[pygame.K_DOWN]: dy = 1
        self.active_player.move(dx, dy)

        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_TAB: # Botón para cambiar de jugador
                    self.switch_player()
                if event.key == pygame.K_x: # Botón de pase
                    if self.ball.carrier == self.active_player:
                        target = self.find_nearest_teammate()
                        self.active_player.pass_ball(self.ball, target)
                if event.key == pygame.K_c: # Botón de marcar / chutar
                    if self.ball.carrier == self.active_player:
                        self.active_player.shoot(self.ball, self.goal_b_rect)
                    else:
                        target = self.find_opponent_with_ball_in_range()
                        if target:
                            self.active_player.tackle(target)

    def switch_player(self):
        # Lógica para encontrar al jugador del equipo humano más cercano al balón
        # y asignarlo como self.active_player
        pass

    def update(self):
        self.space.step(1/60.0)
        self.todos_los_sprites.update()
        # Aquí iría la IA del equipo rival para mover a sus jugadores
Clase 26: Fútbol de Rol en Tiempo Real
Contenidos de la Clase
Discusión sobre el Género Híbrido (20 min): Iniciar la clase presentando el nuevo concepto de "Fútbol de Rol Táctico en Tiempo Real". Discutir cómo se reinterpretan las estadísticas de los jugadores (Creatividad, Anticipación, Temple) en este nuevo contexto de acción directa.

El Balón como Entidad Física (25 min):

Crear la clase Ball como un PhysicsObject.

Implementar la lógica de carrier: cuando un jugador tiene el balón, la posición del balón se fija a la del jugador y sus físicas se "pausan". Cuando se pasa o se chuta, se libera (carrier = None) y el motor de físicas toma el control.

El Jugador Multifacético (40 min):

Refactorizar masivamente la clase SoccerPlayer.

Implementar el movimiento directo con las flechas en el método update o en un nuevo método move.

Añadir el atributo de radio_marcaje basado en la Anticipación y visualizarlo.

Esbozar los métodos de acción: pass_ball, shoot, tackle. Explicar que en la próxima clase se implementará la "resolución" basada en stats de estas acciones.

Orquestando la Acción en Tiempo Real (35 min):

Modificar la MatchScene para que el bucle principal sea en tiempo real.

Implementar la lógica en handle_events para capturar los botones de acción: cambiar jugador, pasar, y el botón contextual de chutar/marcar.

Mostrar cómo el estado del juego ya no es por turnos, sino que depende constantemente de quién es el active_player y quién es el ball.carrier.

Cierre y Próximos Pasos
Resumen: ¡Hemos realizado una transformación radical y emocionante! Hemos evolucionado nuestro juego de un concepto por turnos a un híbrido de acción y rol en tiempo real. Nuestro motor ahora soporta control directo de personajes, acciones contextuales y una interacción mucho más dinámica entre las entidades y la física.

Adelanto de la Clase 27: "Controlamos a nuestro equipo, pero el rival está inmóvil. En la próxima clase, daremos el paso definitivo para crear un partido completo: programaremos la Inteligencia Artificial del equipo rival. Crearemos una IA basada en estados para los jugadores controlados por la máquina: decidirán cuándo presionar, cuándo mantener la posición, y los jugadores ofensivos se desmarcarán para recibir pases, ¡creando un oponente dinámico y desafiante!"






vamos por la nueva clase. Y ten presenta para alguna futura clase, como evaluar quien hizo el mejor insulto poetico, en el duelo, si el defensor respondió bien. podemos pensar capaz mejor que insultos en que digan poesía. si hay duelo, quien ataca dice la poesía, y el defensor la responde, necesitamos que con ia tipo gemini, evalue si el nivel de poesía del atacante es mejor que el defensa, en ese escenario avanza, si el de defensor es mejor le quita la pelota, o que empatn y tengan que seguir el duelo. quiza los jugadores más habilidosos, pueden ser ayudados por la ia para generar mejor poesía, donde se le sugieran palabras o frases. tambien cuando se haga un pase, se deeb permitor que con las flechas se diriga ese pase. seria bueno también que el juez, si pita falta, lo haga con una rima. y que el estadio tenga ambientación sonora, que se pueda habilitar o no, y que lso cantos sea literatuera de alta calidad para aentar a los jugadores




Ver razonamiento
¡Fantástico! Estas son ideas excepcionales que elevan el proyecto a un nivel completamente nuevo de originalidad y creatividad. Olvidemos el simple "Fútbol de Rol Táctico"; estamos a punto de crear el primer "Fútbol Poético: El Duelo Lírico" del mundo.

Me encanta. Esta clase fusionará todo lo que hemos aprendido: la gestión de escenas, el control de personajes en tiempo real, el sistema de diálogos y la potencia de la IA generativa. Abróchate el cinturón.

1. Discusión: La Síntesis Final - Acción, Rol y Poesía
Estamos creando un género híbrido único. Así es como funcionará nuestro nuevo modelo:

Fase de Acción en Tiempo Real:

Controlas a un jugador de tu equipo con las flechas, moviéndote libremente por el campo.

Puedes cambiar al jugador más cercano al balón con una tecla (TAB).

Si tienes el balón, puedes iniciar un pase. Al pulsar el botón de pase, el juego entra en un modo de "apuntado" donde diriges el pase con las flechas antes de soltarlo.

Fase de Duelo Poético (Por Turnos):

Cuando un atacante con el balón entra en el área de influencia (el círculo de Anticipación) de un defensor, el juego en tiempo real se pausa.

Comienza un Duelo Lírico. La cámara se centra en los dos jugadores.

El atacante (sea la IA o el jugador) presenta un verso poético.

El defensor (sea la IA o el jugador) debe responder con otro verso.

Ambos versos se envían a la IA (Gemini), que actúa como un juez literario imparcial.

Resolución por IA:

La IA evalúa los versos basándose en la creatividad, la métrica, la rima y la temática futbolística.

Devuelve un resultado: ATAQUE_GANA, DEFENSA_GANA o EMPATE.

Si el ataque gana: El defensor queda "roto" y el atacante avanza, continuando en tiempo real.

Si la defensa gana: El defensor roba el balón y la posesión cambia.

Si hay empate: Se puede iniciar un segundo verso para desempatar, o el balón queda suelto.

El Toque Artístico:

El árbitro es un bardo que canta las faltas con rimas.

La afición en el estadio no grita cánticos normales, sino que recita fragmentos de Borges, Shakespeare o Cervantes, creando una atmósfera surrealista y culta. El sonido, por supuesto, será opcional.

2. Estructura y Código: La Fusión
Reutilizaremos y adaptaremos gran parte de lo que ya hemos construido. El cambio principal estará en la MatchScene y en cómo usamos nuestro DialogueManager.

engine/dialogue.py (Adaptado para ser el Juez)
La clase es la misma, pero la forma en que la usamos cambia drásticamente. El system_prompt es la clave de todo.

Python

# Un ejemplo del prompt que usaríamos para inicializar el DialogueManager
PROMPT_JUEZ = """
Eres "El Crítico Lirico", un juez de duelos poéticos que ocurren durante un partido de fútbol. Tu criterio es impecable y tu juicio, final.
Te proporcionaré dos versos: un 'VERSO_ATACANTE' y un 'VERSO_DEFENSOR'.
Tu tarea es evaluar cuál es superior basándote en: 1. Creatividad y originalidad. 2. Calidad poética (rima, ritmo, métrica). 3. Relevancia temática con el fútbol, la estrategia o la emoción del momento.

Tu respuesta DEBE ser únicamente una de las siguientes tres palabras clave, seguida de dos puntos y una breve crítica rimada de dos líneas.

Las palabras clave son:
- ATACANTE_GANA
- DEFENSA_GANA
- EMPATE

Ejemplo 1 de respuesta:
DEFENSA_GANA: La réplica fue un muro, precisa y concisa, dejando al atacante sin su anhelada premisa.

Ejemplo 2 de respuesta:
ATACANTE_GANA: Con una metáfora audaz y un ritmo galopante, el verso del rival se ha vuelto irrelevante.

Ejemplo 3 de respuesta:
EMPATE: Ambos versos mostraron gran destreza y valía, el duelo continúa hasta el siguiente día.
"""
soccer_rpg/entities/player.py (Pequeños Añadidos)
Python

# soccer_rpg/entities/player.py
# ... (código existente) ...

class SoccerPlayer(GameObject):
    # ...
    def get_poetic_line(self, is_attacking=True):
        """
        Genera una línea de poesía base.
        En el futuro, la IA podría ayudar aquí para jugadores con alta Creatividad.
        """
        if is_attacking:
            # Lógica para generar o seleccionar una línea de ataque
            return "Mis pies danzan sobre el césped, un vals imprevisto,"
        else:
            # Lógica para generar o seleccionar una línea de defensa
            return "Pero tu danza termina donde mi pie ha pisado."
soccer_rpg/scenes/match_scene.py (Grandes Cambios)
Esta escena ahora gestionará una máquina de estados mucho más compleja.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.game_state = "REAL_TIME"
        # Estados: REAL_TIME, AIMING_PASS, POETRY_DUEL, WAITING_FOR_AI

        # Inicializamos el juez de IA
        self.juez_ia = DialogueManager(API_KEY, PROMPT_JUEZ)
        # ...

    def update(self):
        if self.game_state == "REAL_TIME":
            self.todos_los_sprites.update()
            # Comprobar si un atacante con balón entra en el radio de un defensor
            attacker = self.ball.carrier
            if attacker:
                for defender in self.get_opponent_team(attacker.team):
                    if pygame.sprite.collide_circle(attacker, defender, defender.radio_marcaje):
                        self.initiate_poetry_duel(attacker, defender)
                        break

        elif self.game_state == "POETRY_DUEL":
            # La lógica de esta UI esperaría el input del jugador
            pass

        elif self.game_state == "WAITING_FOR_AI":
            # Podríamos mostrar un icono de "pensando..."
            pass

    def handle_events(self, events):
        if self.game_state == "REAL_TIME":
            # ... (lógica de movimiento y botones de acción) ...
            if keys[pygame.K_x] and self.ball.carrier == self.active_player:
                self.game_state = "AIMING_PASS"

        elif self.game_state == "AIMING_PASS":
            # ... (lógica para dirigir el pase con flechas) ...
            # Al soltar el botón, se ejecuta el pase y se vuelve a REAL_TIME
            pass

        elif self.game_state == "POETRY_DUEL":
            # ... (lógica para que el jugador escriba su verso y pulse Enter) ...
            # Al pulsar Enter, se envía a la IA:
            # self.submit_duel_to_ai(attacker_verse, defender_verse)
            pass

    def initiate_poetry_duel(self, attacker, defender):
        self.game_state = "POETRY_DUEL"
        self.duel_participants = (attacker, defender)
        print(f"¡Duelo Poético entre {attacker.nombre} y {defender.nombre}!")
        # Aquí se mostraría la UI del duelo

    def submit_duel_to_ai(self, attacker_verse, defender_verse):
        self.game_state = "WAITING_FOR_AI"
        prompt_final = f"VERSO_ATACANTE: '{attacker_verse}'\nVERSO_DEFENSOR: '{defender_verse}'"

        # Esto debería hacerse en un hilo secundario para no congelar el juego,
        # pero para simplificar lo hacemos de forma directa.
        response = self.juez_ia.send_message(prompt_final)
        self.resolve_duel_result(response)

    def resolve_duel_result(self, result):
        # ... (lógica para procesar ATACANTE_GANA, DEFENSA_GANA o EMPATE) ...
        self.game_state = "REAL_TIME"

    def draw(self, surface):
        # ...
        if self.game_state == "POETRY_DUEL":
            # Dibujar la UI específica del duelo
            pass
Clase 27: El Duelo Lírico y el Juez de IA
Contenidos de la Clase
Discusión sobre la Jugabilidad Híbrida (20 min): Presentar el concepto final de "Fútbol Poético". Discutir cómo se alternará entre una fase de acción en tiempo real (movimiento, pases) y una fase por turnos (los duelos poéticos), y cómo esta combinación crea un ritmo de juego único.

El Juez de IA: Diseño de Prompts Avanzado (35 min):

El foco principal de la clase. Presentar el PROMPT_JUEZ.

Desglosar sus componentes: el rol, el contexto, la tarea y, lo más importante, el formato de salida estricto. Explicar por qué forzar a la IA a responder solo con PALABRA_CLAVE: crítica es esencial para que nuestro código pueda interpretar el resultado de forma fiable.

Realizar pruebas en vivo enviando versos de ejemplo al DialogueManager para ver las respuestas del juez.

Implementando la Máquina de Estados Híbrida (40 min):

Refactorizar la MatchScene para incluir la nueva máquina de estados: REAL_TIME, AIMING_PASS, POETRY_DUEL, WAITING_FOR_AI.

Implementar la lógica de update que detecta cuándo debe iniciarse un duelo (un jugador con balón dentro del radio de un defensor).

Implementar el handle_events para el pase dirigido con flechas, una mecánica en tiempo real.

Ambientación Artística (15 min):

Usando nuestro AudioManager, mostrar cómo cargar una lista de "cánticos literarios" y reproducirlos de forma aleatoria de fondo.

Crear una función simple para el árbitro que, al llamarla, elija una de varias rimas preescritas para señalar una falta.

Cierre y Próximos Pasos
Resumen: Hoy hemos alcanzado un nuevo pico de creatividad y complejidad técnica. Hemos diseñado un juego híbrido único que mezcla acción y estrategia por turnos, y hemos implementado un juez de IA capaz de tomar decisiones subjetivas basadas en la calidad poética. Nuestro motor ha demostrado ser una plataforma increíblemente flexible para las ideas más audaces.

Adelanto y Conclusión del Módulo: "Hemos construido el sistema de juicio y el flujo del partido. En la clase final de este módulo, nos centraremos en la experiencia del jugador. Implementaremos la ayuda de la IA como musa para los jugadores con alta Creatividad, construiremos una interfaz de usuario pulida para los duelos, y añadiremos el duelo final contra el portero para marcar un gol. Con esto, cerraremos el ciclo y tendremos un juego de 'Fútbol Poético' completo y funcional, listo para ser empaquetado."






vamos por la próxima clase


¡Excelente! Llegamos a la culminación de nuestro proyecto más creativo hasta la fecha. Es hora de pulir la experiencia, implementar las mecánicas finales y convertir nuestro prototipo de "Fútbol Poético" en un juego completo y satisfactorio.

En esta clase, nos centraremos en tres áreas clave: la ayuda de la IA como musa para el jugador, la implementación del tiro a puerta como un duelo lírico final, y la creación de una interfaz de usuario (UI) clara y atractiva para los duelos.

1. Discusión: La Experiencia del Jugador (UX/UI)
Un juego puede tener mecánicas brillantes, pero si la forma de interactuar con ellas es confusa o frustrante, el jugador lo abandonará. Una buena interfaz de usuario (UI) y experiencia de usuario (UX) son cruciales.

Claridad y Feedback: El jugador debe entender en todo momento qué está pasando, qué se espera de él y cuál es el resultado de sus acciones. Nuestra UI para el duelo debe mostrar claramente quién ataca, quién defiende, el verso del rival y las opciones de respuesta.

La Musa de la IA (Asistencia al Jugador): ¿Qué pasa si el jugador no es un poeta nato? Forzarlo a escribir desde cero puede ser intimidante. Aquí es donde la IA puede actuar como una musa inspiradora. Para los jugadores con alta Creatividad, podemos ofrecer un botón de "sugerencia" que llame a la IA con un prompt diferente para que genere una o varias líneas poéticas, dándole al jugador una base sobre la que trabajar o la opción de usarla directamente. Esto es un diseño de UX excelente: reduce la fricción sin eliminar el desafío.

Momentos de Clímax: El tiro a puerta no puede ser un duelo más. Debe sentirse como el clímax de la jugada. La interfaz puede cambiar, la música puede volverse más tensa, y el desafío poético contra el portero debe ser el más difícil de todos, utilizando su estadística de Temple.

Ejemplos en juegos conocidos:

Phoenix Wright: Ace Attorney ⚖️: Este juego es una clase magistral de UI para duelos verbales. Cuando presentas evidencia, hay animaciones y sonidos impactantes ("¡Protesto!"). La interfaz muestra claramente las contradicciones y guía al jugador. Nuestra UI del duelo se inspirará en esta claridad.

Slay the Spire 🃏: Un juego de construcción de mazos donde la UI es perfecta. Muestra claramente la "intención" del enemigo (si va a atacar o a defender), permitiendo al jugador planificar su turno. De forma similar, nuestra "musa de IA" le dará al jugador una idea de una posible "intención" poética.

Juegos con "Quick Time Events" (QTE): Aunque a veces criticados, los QTE son un ejemplo de cómo la UI se enfoca en un momento de clímax. Cuando vas a tirar a puerta, la pantalla se puede centrar, los bordes pueden oscurecerse, y toda la atención se centra en el duelo final.

2. Estructura de Archivos Actualizada
Añadimos un módulo ui_elements.py a nuestro motor para crear componentes de UI reutilizables, como botones.

/curso_pygame/
|
|-- engine/
|   |-- ... (archivos existentes)
|   |-- ui_elements.py       # ¡NUEVO! Para botones y otros elementos.
|
|-- soccer_rpg/
|   |-- ... (resto de la estructura)
|
|-- requirements.txt
3. El Código del Motor Actualizado
engine/ui_elements.py (¡NUEVO!)
Python

# engine/ui_elements.py
import pygame

class Button:
    def __init__(self, x, y, width, height, text, callback):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.callback = callback
        self.font = pygame.font.Font(None, 32)
        self.color = (100, 100, 120)
        self.hover_color = (150, 150, 170)

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.callback()

    def draw(self, surface):
        color = self.hover_color if self.rect.collidepoint(pygame.mouse.get_pos()) else self.color
        pygame.draw.rect(surface, color, self.rect, border_radius=5)

        text_surf = self.font.render(self.text, True, (255, 255, 255))
        text_rect = text_surf.get_rect(center=self.rect.center)
        surface.blit(text_surf, text_rect)
engine/dialogue.py (Pequeño Añadido)
Añadimos un método para la "musa".

Python

# engine/dialogue.py
# ... (clase DialogueManager existente) ...
class DialogueManager:
    # ...
    def get_poetic_suggestion(self, topic):
        """Pide a la IA una sugerencia poética sobre un tema."""
        # Usamos un modelo separado o un prompt diferente para no contaminar la conversación del juez
        try:
            # Este es un ejemplo simple, podría ser más complejo
            prompt = f"Escribe un verso poético corto y contundente sobre el siguiente tema de fútbol: '{topic}'. El verso debe ser inspirador para un jugador."
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Error en la API de sugerencia: {e}")
            return "El balón... es redondo."
4. El Código del Juego Actualizado
soccer_rpg/scenes/match_scene.py (Grandes Cambios)
Integramos la UI del duelo, la musa de IA y el tiro a puerta.

Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...
from engine.ui_elements import Button

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.duel_ui_elements = []
        self.setup_duel_ui()

    def setup_duel_ui(self):
        """Crea los elementos de la UI para el duelo, pero los mantiene inactivos."""
        self.duel_input_box = pygame.Rect(100, 400, 600, 40)
        self.duel_input_text = ""

        # Botón para la musa de IA
        self.suggestion_button = Button(300, 450, 200, 40, "Pedir Musa", self.on_click_suggestion)
        self.duel_ui_elements.append(self.suggestion_button)

    def on_click_suggestion(self):
        """Callback para el botón de la musa."""
        if self.game_state == "POETRY_DUEL" and self.active_player.team == "A": # Si es el turno del jugador humano
            attacker = self.duel_participants[0]
            if attacker.creatividad > 85: # Solo los jugadores muy creativos pueden pedir ayuda
                # El tema podría ser más dinámico
                suggestion = self.juez_ia.get_poetic_suggestion("un regate imparable")
                self.duel_input_text = suggestion
            else:
                print("Este jugador confía en su propia inspiración.")

    def handle_events(self, events):
        # ...
        if self.game_state == "POETRY_DUEL":
            for element in self.duel_ui_elements:
                element.handle_event(event)
            # ... (lógica para escribir en el input box) ...

    def initiate_poetry_duel(self, attacker, defender):
        # ...
        # Lógica para el tiro a puerta
        is_shot_on_goal = self.is_in_shooting_range(attacker)
        if is_shot_on_goal:
            self.game_state = "SHOOTOUT_DUEL"
            print("¡DUELO FINAL CONTRA EL PORTERO!")
        else:
            self.game_state = "POETRY_DUEL"
        # ...

    def draw(self, surface):
        # ... (dibujado del campo y jugadores) ...

        if self.game_state in ["POETRY_DUEL", "SHOOTOUT_DUEL"]:
            # Dibujar un fondo semitransparente para enfocar la acción
            overlay = pygame.Surface((surface.get_width(), surface.get_height()), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 180))
            surface.blit(overlay, (0, 0))

            # Dibujar la UI del duelo
            # ... (textos del atacante, caja de input, etc.) ...
            for element in self.duel_ui_elements:
                element.draw(surface)
Clase 28: Experiencia de Usuario y Clímax del Juego
Contenidos de la Clase
Discusión sobre UX/UI en Duelos (20 min): Iniciar con la discusión sobre la importancia de la claridad y el feedback en las interfaces de duelo, usando los ejemplos de Phoenix Wright y Slay the Spire.

La Musa de IA (35 min):

Implementar el método get_poetic_suggestion en el DialogueManager. Explicar que es buena idea usar un prompt diferente al del juez para esta tarea.

Crear la clase Button en el motor como un componente de UI reutilizable.

En MatchScene, añadir el botón "Pedir Musa". Implementar la lógica para que solo los jugadores con alta Creatividad puedan usarlo, conectando directamente las estadísticas del personaje con las mecánicas de la UI.

El Duelo Final: Tiro a Puerta (40 min):

Definir una "zona de tiro" en el campo.

En initiate_poetry_duel, comprobar si el atacante está en esa zona. Si es así, el estado del juego cambia a SHOOTOUT_DUEL y el oponente es siempre el portero rival.

Discutir cómo la resolución de este duelo final dependería del Temple de ambos jugadores, haciendo que la llamada a la IA o la lógica de resolución sea más difícil.

Pulido de la Interfaz de Duelo (25 min):

Implementar la UI final para los duelos en el método draw de MatchScene.

Mostrar cómo usar un overlay semitransparente para oscurecer el fondo y centrar la atención del jugador en el duelo.

Organizar el texto del atacante, la caja de entrada del jugador y los botones de forma clara y legible.

Cierre y Conclusión Final del Curso
Resumen General: ¡Lo hemos conseguido! A lo largo de este curso, hemos construido un motor de juegos increíblemente versátil desde cero. Hemos creado cuatro juegos completos de géneros totalmente distintos: un Arkanoid, un plataformero, un juego de físicas y, finalmente, nuestro innovador "Fútbol Poético". Hemos aprendido sobre gestión de escenas, físicas, IA de patrulla, sistemas de diálogo, y hoy, hemos culminado nuestro viaje integrando IA generativa de una forma creativa y profunda, no como un truco, sino como una mecánica central.

El Siguiente Paso es Tuyo: Has pasado de ser un estudiante a ser un diseñador y desarrollador de juegos. Tienes las herramientas, el conocimiento y, lo más importante, la experiencia para tomar cualquier idea, por extraña o ambiciosa que sea, y convertirla en una realidad jugable. El motor es tuyo para expandir, y el mundo de los videojuegos está esperando tus creaciones.

¡Enhorabuena, has completado el curso! Ha sido un placer guiarte en este viaje. Ahora, ¡ve y crea!






incluiste como generar lso cantos de la hincahcda usando l iteratura clasica?


¡Tienes toda la razón! Es una de las ideas más originales y se me pasó incluirla en el código de la última clase. Mis disculpas. Es un detalle de ambientación fantástico que merece ser implementado.

Añadir los cánticos literarios es un proceso muy similar al que usamos para el juez o la musa, pero más simple. No necesitamos una respuesta, solo generar contenido.

Aquí te muestro cómo lo integraríamos.

1. El Prompt para el "Bardo de la Grada"
Primero, necesitamos un nuevo prompt para una instancia de DialogueManager que se encargará de generar los cánticos. Este prompt le dará a la IA el rol de una hinchada increíblemente culta.

Python

# Prompt para el generador de cánticos
PROMPT_HINCHADA = """
Eres el 'Coro de los Eruditos', la hinchada de un equipo de fútbol poético. Tu propósito es alentar a tu equipo, pero en lugar de cánticos vulgares, recitas fragmentos o imitaciones del estilo de grandes autores de la literatura universal.

Tu tarea es generar un cántico corto (2 o 3 líneas) que sea apropiado para un momento de tensión o aliento en un partido de fútbol. El estilo debe ser elevado y dramático.

Ejemplo de estilo de Borges: "En el laberinto del campo, un solo hombre es el minotauro, y el balón, su hilo de Ariadna."
Ejemplo de estilo de Shakespeare: "¡Oh, valiente delantero! Que tu disparo vuele certero, pues la duda es la traidora que nos hace perder lo bueno que a menudo podríamos ganar."
Ejemplo de estilo de Cervantes: "Acometed, nobles jugadores, la aventura de este gol, que los gigantes que veis ahí no son sino molinos de viento defensivos."

Genera un nuevo cántico original con este estilo.
"""
2. Integración en el AudioManager
La forma más elegante de manejar esto es expandir nuestro AudioManager para que no solo reproduzca sonidos pregrabados, sino que también pueda gestionar una "banda sonora" de cánticos generados.

engine/audio.py (Actualizado)
Python

# engine/audio.py
import pygame
import random
from .dialogue import DialogueManager # Necesitaremos el DialogueManager aquí
from adventure.config import API_KEY # Y la API Key

class AudioManager:
    def __init__(self, num_channels=16):
        # ... (código de inicialización existente) ...

        # ¡NUEVO! Generador de cánticos
        self.chant_generator = DialogueManager(API_KEY, PROMPT_HINCHADA)
        self.chants = [] # Cache para guardar los cánticos generados
        self.chant_font = pygame.font.Font(None, 22)
        self.current_chant_surface = None
        self.chant_timer = 0
        self.chant_cooldown = 10 * 60 # 10 segundos (a 60 FPS)

    def update_chants(self, surface):
        """Gestiona la generación y visualización de cánticos."""
        self.chant_timer += 1
        if self.chant_timer > self.chant_cooldown:
            self.chant_timer = 0
            # Pedimos un nuevo cántico a la IA
            # Esto debería ir en un hilo para no congelar el juego
            print("Generando nuevo cántico...")
            new_chant_text = self.chant_generator.send_message("Genera un nuevo cántico de aliento.")
            self.chants.append(new_chant_text)

            # Creamos la superficie de texto para el último cántico
            self.current_chant_surface = self.chant_font.render(f"La grada canta: '{new_chant_text}'", True, (200, 200, 150, 200))

        # Dibujamos el cántico actual si existe
        if self.current_chant_surface:
            chant_rect = self.current_chant_surface.get_rect(centerx=surface.get_width() / 2, y=surface.get_height() - 20)
            surface.blit(self.current_chant_surface, chant_rect)

    # ... (resto de los métodos existentes: load_sound, play_sound, etc.) ...
3. Implementación en la MatchScene
Ahora, en lugar de que la escena gestione el audio, simplemente le decimos al AudioManager que se actualice en cada fotograma.

soccer_rpg/scenes/match_scene.py (Actualizado)
Python

# soccer_rpg/scenes/match_scene.py
# ... (imports) ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        super().__init__(scene_manager)
        # ...
        # El audio_manager se crea aquí
        self.audio_manager = AudioManager()
        self.sound_enabled = True # El jugador puede cambiar esto
        # ...

    def handle_events(self, events):
        # ...
        for event in events:
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_m: # Tecla 'M' para mutear/desmutear
                    self.sound_enabled = not self.sound_enabled
                    if not self.sound_enabled:
                        pygame.mixer.music.stop()
                    else:
                        self.audio_manager.play_music() # Asumiendo que hay música de fondo

    def update(self):
        # ... (toda la lógica de juego) ...
        pass

    def draw(self, surface):
        # ... (dibujado del campo, jugadores, UI del duelo, etc.) ...

        # Al final del dibujado, actualizamos y dibujamos los cánticos
        if self.sound_enabled:
            self.audio_manager.update_chants(surface)
Resumen de la Implementación
Rol Definido: Creamos un PROMPT_HINCHADA que le da a la IA una personalidad y tarea muy específicas: ser una afición literaria.

Gestor Centralizado: Modificamos el AudioManager para que contenga una instancia del DialogueManager. Ahora, el AudioManager es responsable de todo el paisaje sonoro, incluyendo estos cánticos generados.

Generación Periódica: En el update_chants del AudioManager, usamos un temporizador. Cada cierto tiempo (ej. 10-15 segundos), se hace una llamada a la API para generar un nuevo cántico.

Nota importante: Como se menciona en el código, en un juego real, esta llamada a la API (que puede tardar 1-2 segundos) se haría en un hilo de ejecución secundario (threading) para no congelar el juego. Para nuestro curso, la implementación directa es suficiente para demostrar el concepto.

Visualización Sutil: El cántico generado se renderiza como un texto que aparece en la parte inferior de la pantalla, como si fuera un subtítulo del ambiente, sin ser intrusivo.

Control del Jugador: Añadimos una simple tecla (M) que permite al jugador activar o desactivar toda la ambientación sonora (música y cánticos), respetando sus preferencias.

Con esta adición, el juego adquiere una capa de atmósfera increíblemente única y memorable. La idea de que los cánticos de la grada sean generados por IA y basados en literatura clásica es exactamente el tipo de detalle que hace que un juego indie destaque. ¡Excelente sugerencia!






yo quiero que la hinchada cante, como podemos epandir e motor para que tome el texto generado y lo convierta en cantico?



Ver razonamiento
¡Excelente pregunta! Darle voz a la hinchada es el siguiente paso lógico para una inmersión total. Esto nos adentra en el fascinante mundo de la síntesis de voz (Text-to-Speech o TTS).

Vamos a expandir nuestro motor para que no solo genere el texto del cántico, sino que lo convierta en un archivo de audio y lo reproduzca durante el partido, dándole una voz real a nuestro "Coro de los Eruditos".

1. Discusión: Dando Voz a la IA - Text-to-Speech en Juegos
La síntesis de voz es la tecnología que convierte texto escrito en habla audible. En los videojuegos, tiene aplicaciones muy poderosas:

Accesibilidad: Es una herramienta fundamental para que jugadores con discapacidad visual puedan disfrutar de juegos con mucho texto.

Prototipado Rápido: Los desarrolladores pueden usar TTS para escuchar los diálogos en el contexto del juego antes de contratar y grabar a actores de voz, lo cual es muy costoso.

Narración Dinámica: Para juegos con texto generado proceduralmente (como el nuestro), el TTS es la única forma de darle voz a un contenido que no existe hasta que se crea.

Diseño de Personajes: Una voz robótica o sintetizada puede ser una elección de diseño para personajes que son androides, inteligencias artificiales o entidades sobrenaturales.

Ejemplos en juegos conocidos:

Portal 2 🤖: Las voces icónicas de GLaDOS y Wheatley, aunque interpretadas por actores, se procesaron masivamente para sonar como voces de IA, inspirándose en la cadencia y el timbre del TTS para crear su personalidad.

System Shock 👾: La voz de la malévola IA SHODAN es uno de los personajes más memorables de la historia de los videojuegos. Su voz entrecortada, con fallos y un tono sintético, es puro diseño de sonido basado en los principios del TTS para crear una sensación de terror tecnológico.

Moonbase Alpha 🧑‍🚀: Este juego de la NASA se hizo viral por su sistema de chat con TTS, que producía frases hilarantes y robóticas ("John Madden! aeiou"). Demuestra cómo incluso un TTS básico puede crear una experiencia memorable.

Nuestra Herramienta: gTTS (Google Text-to-Speech)

Usaremos la librería gTTS de Python. Es muy fácil de usar y utiliza el potente motor de síntesis de voz de Google para generar archivos de audio de alta calidad.

2. Estructura de Archivos y Dependencias Actualizadas
Añadimos la nueva librería a nuestros requerimientos. La estructura de archivos no necesita cambiar.

requirements.txt (Actualizado)
Plaintext

pygame
pyinstaller
pymunk
google-generativeai
gTTS
(No olvides ejecutar pip install -r requirements.txt o pip install gTTS)

3. El Código del Motor Actualizado
Aquí viene el cambio más importante y conceptualmente más avanzado que hemos hecho. Las llamadas a internet (tanto para Gemini como para gTTS) son lentas. Si las hacemos en el bucle principal, el juego se congelará durante varios segundos. Para evitarlo, debemos ejecutarlas en un hilo de ejecución secundario (threading).

engine/audio.py (Grandes Cambios)
Python

# engine/audio.py
import pygame
import random
import threading # ¡Importamos la librería para hilos!
import os
from gtts import gTTS # ¡Importamos gTTS!
from .dialogue import DialogueManager
from adventure.config import API_KEY

class AudioManager:
    def __init__(self, num_channels=16):
        # ... (código de inicialización existente) ...
        self.chant_generator = DialogueManager(API_KEY, PROMPT_HINCHADA)
        self.chant_cooldown = 15 * 60 # 15 segundos
        self.chant_timer = self.chant_cooldown # Para que genere uno al empezar

        # Variables para manejar el hilo
        self.chant_thread = None
        self.chant_audio_path = "temp_chant.mp3"
        self.new_chant_ready = False

    def _generate_and_synthesize_chant(self):
        """
        Esta función se ejecutará en un hilo separado.
        Contiene las operaciones lentas (llamadas a internet y escritura de archivo).
        """
        print("Hilo de cánticos: Generando texto con IA...")
        try:
            # 1. Generar el texto del cántico
            chant_text = self.chant_generator.send_message("Genera un nuevo cántico de aliento.")
            if not chant_text: return

            print(f"Hilo de cánticos: Texto recibido - '{chant_text[:30]}...'")

            # 2. Sintetizar el texto a un archivo MP3
            tts = gTTS(text=chant_text, lang='es', slow=False)
            tts.save(self.chant_audio_path)

            print(f"Hilo de cánticos: Archivo de audio '{self.chant_audio_path}' guardado.")
            self.new_chant_ready = True # Marcamos que el nuevo cántico está listo para ser reproducido
        except Exception as e:
            print(f"Error en el hilo de cánticos: {e}")

    def update_chants(self):
        """Gestiona el ciclo de vida de los cánticos."""
        self.chant_timer += 1

        # Si hay un cántico nuevo listo Y la música no está sonando, lo reproducimos
        if self.new_chant_ready and not pygame.mixer.music.get_busy():
            print("Reproduciendo nuevo cántico...")
            try:
                pygame.mixer.music.load(self.chant_audio_path)
                pygame.mixer.music.play()
                self.new_chant_ready = False
                # Podríamos eliminar el archivo aquí si quisiéramos
                # os.remove(self.chant_audio_path)
            except pygame.error as e:
                print(f"No se pudo reproducir el cántico: {e}")

        # Si ha pasado el tiempo de espera Y el hilo anterior ha terminado, iniciamos uno nuevo
        if self.chant_timer > self.chant_cooldown and (self.chant_thread is None or not self.chant_thread.is_alive()):
            self.chant_timer = 0
            # Creamos y lanzamos el hilo
            self.chant_thread = threading.Thread(target=self._generate_and_synthesize_chant)
            self.chant_thread.start()

    # ... (resto de métodos sin cambios) ...

soccer_rpg/scenes/match_scene.py (Mínimos Cambios)
Gracias a nuestro motor bien diseñado, la escena del juego apenas necesita saber que esto está ocurriendo.

Python

# soccer_rpg/scenes/match_scene.py
# ...

class MatchScene(Scene):
    # ...
    def update(self):
        # ... (lógica de juego existente) ...

        # Simplemente le decimos al gestor de audio que se actualice
        if self.sound_enabled:
            self.audio_manager.update_chants()

    def draw(self, surface):
        # Ya no necesitamos dibujar el texto del cántico aquí. El audio lo es todo.
        # ... (código de dibujado existente) ...
        pass
Clase 29: El Sonido de la IA - Síntesis de Voz y Concurrencia
Contenidos de la Clase
Discusión sobre Voz Sintética (20 min): Iniciar con la discusión sobre TTS, sus aplicaciones en juegos y los ejemplos mencionados, desde la accesibilidad hasta el diseño de personajes icónicos.

Introducción a gTTS (25 min):

Instalar la librería (pip install gTTS).

Mostrar un ejemplo simple y aislado: escribir un script corto que tome un texto, lo guarde como hola.mp3 y luego lo reproduzca con Pygame. Esto demuestra el flujo básico.

El Problema de la Concurrencia (35 min):

Ejecutar una llamada a la API de Gemini o a gTTS.save() directamente en el bucle update y mostrar cómo el juego se congela por completo.

Explicar por qué ocurre esto: el bucle principal debe esperar a que termine la lenta operación de red/disco.

Introducir threading como la solución. Explicar el concepto de un hilo de ejecución como un "mini-programa" que corre en paralelo al bucle principal sin bloquearlo.

Implementando el AudioManager Asíncrono (40 min):

Refactorizar el AudioManager paso a paso para crear la nueva lógica.

Crear la función _generate_and_synthesize_chant que contiene las operaciones lentas.

En update_chants, implementar el sistema de control:

Comprobar el cooldown.

Comprobar si el hilo anterior ha terminado (is_alive()).

Lanzar el nuevo hilo.

Comprobar si hay un nuevo archivo de audio listo para reproducir (new_chant_ready).

Cierre y Próximos Pasos
Resumen: Hemos dado un paso gigantesco en la complejidad y profesionalidad de nuestro motor. No solo generamos contenido con IA, sino que le damos voz en tiempo real sin sacrificar el rendimiento del juego. Hemos aprendido sobre síntesis de voz y hemos implementado una solución de concurrencia con hilos, una técnica avanzada indispensable para cualquier aplicación moderna.

Adelanto Final del Curso: "Con la capacidad de crear diálogos, juzgar poesía y ahora generar audio dinámicamente, nuestro motor está completo. La última etapa de nuestro viaje será la conclusión de los proyectos. En las clases finales, nos enfocaremos en pulir la jugabilidad de nuestros tres juegos, añadiendo los menús finales, pantallas de victoria/derrota, y empaquetando cada uno de ellos para que tengas un portafolio de tres juegos únicos y terminados, listos para mostrar al mundo."







avancemos!


¡Vamos allá! Es hora de atar todos los cabos sueltos y llevar nuestros proyectos a la línea de meta.

En esta clase final, nos enfocaremos en la conclusión y el pulido. El objetivo es tomar los prototipos funcionales que hemos construido y añadirles las características que los convierten en experiencias de juego completas: menús, condiciones de victoria claras, y un flujo de juego que guíe al jugador de principio a fin.

1. Discusión: ¿Qué Significa "Terminar" un Juego?
Un juego está "terminado" no cuando se le añaden todas las características posibles, sino cuando el bucle de jugabilidad principal es sólido y satisfactorio y el jugador puede experimentar el juego completo sin interrupciones ni callejones sin salida.

El Bucle de Jugabilidad (Gameplay Loop): Es la secuencia de acciones que el jugador repite.

En nuestro Plataformero: Saltar y moverse -> Recoger monedas -> Evitar enemigos -> Llegar al final del nivel.

En nuestro Fútbol Poético: Moverse en tiempo real -> Iniciar un duelo -> Resolver el duelo -> Ver la consecuencia táctica -> Repetir hasta el gol.

La Estructura de la Experiencia (Player Journey): ¿Cómo guiamos al jugador?

Menú Principal: La puerta de entrada. Debe ser claro y ofrecer las opciones básicas ("Jugar", "Salir").

El Juego en Sí: El núcleo de la experiencia.

Condiciones de Fin: Debe haber una forma clara de ganar (llegar al final, marcar 3 goles) o perder (quedarse sin vidas, recibir 3 goles).

Pantalla Final: Una pantalla de "Victoria" o "Game Over" que da cierre a la sesión y, idealmente, ofrece la opción de volver al menú para jugar de nuevo.

Ejemplos en juegos conocidos:

Cualquier juego de Arcade clásico (Pac-Man, Donkey Kong): Su estructura es la definición de un bucle completo. Tienen una pantalla de título (Menú), el juego principal donde el objetivo es claro (comer todos los puntos, salvar a la princesa), condiciones de fin claras (perder todas las vidas), y una pantalla de "Game Over" con la puntuación, que te incita a volver a intentarlo.

Journey 🏜️: Un ejemplo magistral de viaje del jugador. Comienza sin menús, directamente en el juego. El viaje tiene un principio, un desarrollo y un final narrativo muy claro y emotivo. Al terminar, vuelve suavemente a la pantalla de inicio, invitando a la rejugabilidad.

2. Estructura y Código: Aplicando el Pulido
No añadiremos nuevos módulos al motor. En su lugar, aplicaremos los que ya tenemos (SceneManager, UI, Button) a nuestros proyectos para construir las piezas que faltan.

Proyecto 1: Plataformero - La Meta Final
Objetivo: Añadir una "zona de victoria" al final del nivel.

platformer/entities.py (¡NUEVO!)
Python

# platformer/entities.py
from engine.game_object import GameObject

# ... (clases Player y Platform) ...

class Goal(GameObject):
    """Un objeto que representa el final del nivel."""
    def __init__(self, x, y):
        # Podríamos usar una imagen de una bandera, un portal, etc.
        super().__init__(x, y, 'platformer/assets/goal.png')
platformer/scenes/game_scene.py (Actualizado)
Python

# platformer/scenes/game_scene.py
# ...
from ..entities import Goal
from .victory_scene import VictoryScene # Necesitaremos crear esta escena

class GameScene(Scene):
    def __init__(self, scene_manager):
        # ...
        # Al final de la carga del nivel, añadimos la meta
        self.goal = Goal(1500, 352) # Ponerlo al final del nivel
        self.todos_los_sprites.add(self.goal)
        # ...

    def update(self):
        # ...
        # Comprobar si el jugador ha llegado a la meta
        if pygame.sprite.collide_rect(self.jugador, self.goal):
            self.scene_manager.go_to(VictoryScene, score=self.puntuacion)
Proyecto 2: Fútbol Poético - El Marcador y el Gol
Objetivo: Implementar un marcador, detectar goles y terminar el partido.

soccer_rpg/scenes/match_scene.py (Actualizado)
Python

# soccer_rpg/scenes/match_scene.py
# ...

class MatchScene(Scene):
    def __init__(self, scene_manager):
        # ...
        self.score = {"A": 0, "B": 0}
        self.goal_a_rect = pygame.Rect(0, 250, 50, 100) # Portería izquierda
        self.goal_b_rect = pygame.Rect(750, 250, 50, 100) # Portería derecha
        # ...

    def update(self):
        # ...
        # En la fase de tiempo real, comprobar si el balón entra en una portería
        if self.ball.rect.colliderect(self.goal_a_rect):
            self.score["B"] += 1
            print("¡GOL DEL EQUIPO B!")
            self.reset_positions_after_goal()
        elif self.ball.rect.colliderect(self.goal_b_rect):
            self.score["A"] += 1
            print("¡GOL DEL EQUIPO A!")
            self.reset_positions_after_goal()

        # Comprobar condición de victoria
        if self.score["A"] >= 3 or self.score["B"] >= 3:
            # self.scene_manager.go_to(VictoryScene, score=self.score)
            pass

    def reset_positions_after_goal(self):
        """Reinicia la posición de los jugadores y el balón al centro."""
        # ... (lógica para recolocar a todos los jugadores y al balón) ...
        self.game_state = "REAL_TIME"

    def draw(self, surface):
        # ...
        # Usar nuestra UI del motor para dibujar el marcador
        # self.ui.draw_score(surface, self.score)
        pass
Clase 30: Conclusión de Proyectos y Pulido Final
Contenidos de la Clase
Discusión sobre el "Cierre" (20 min): Iniciar con la discusión sobre la importancia de tener un bucle de jugabilidad y un viaje del jugador completos. Usar los ejemplos de juegos arcade y Journey para ilustrar el concepto.

Implementando Menús y Escenas Finales (35 min):

Crear escenas genéricas de "Menú Principal" y "Pantalla de Victoria/Derrota" usando los componentes de nuestro motor (Scene, Button, UI).

Repasar el SceneManager y cómo se usa para conectar el flujo: Menú -> Juego -> Pantalla Final -> Menú.

Pulido del Plataformero (30 min):

Añadir la entidad Goal al final del nivel del plataformero.

Implementar la lógica de colisión en GameScene que detecta cuando el jugador toca la meta y transiciona a la pantalla de victoria, pasándole la puntuación final.

Pulido del Fútbol Poético (35 min):

Implementar el marcador y las porterías como pygame.Rect.

En el update de la MatchScene, añadir la lógica que comprueba si el balón colisiona con una de las porterías.

Si hay gol, actualizar el marcador, llamar a una función reset_positions_after_goal, y continuar el partido.

Añadir una condición de victoria (ej: el primero que marque 3 goles) que transicione a una pantalla final.

Cierre Final y Despedida del Curso
Resumen Definitivo: Hemos llegado al final de nuestro increíble viaje. No solo hemos aprendido a programar, sino que hemos aprendido a pensar como diseñadores de juegos. Hemos construido un motor robusto y, lo que es más importante, lo hemos utilizado para crear y terminar múltiples juegos, cada uno con su propia personalidad y mecánicas.

Tu Portafolio: Ahora tienes un portafolio tangible con tres proyectos únicos (Plataformero, Fútbol Poético, y el juego de Físicas/Billar). Cada uno demuestra un conjunto diferente de habilidades técnicas y creativas. Tienes el conocimiento para empaquetarlos con PyInstaller y mostrarlos con orgullo.

El Futuro es Tuyo: El desarrollo de juegos es un viaje de aprendizaje constante. Has construido una base sólida que te permitirá explorar cualquier género, cualquier mecánica y cualquier idea que se te ocurra. Has aprendido a aprender.













